In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

from pathlib import Path

# Localizar el dataset
carpeta = Path("data")

if not carpeta.exists():
    carpeta = Path("../data")

df = pd.read_csv(
    carpeta / "SGSST_auditorias_sinteticas_5000.csv"
)

print("Dataset cargado correctamente")
print("Registros disponibles:", len(df))

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nPrimeros registros:")
display(df.head())

Dataset cargado correctamente
Registros disponibles: 5000

Columnas disponibles:
['id_hallazgo', 'id_auditoria', 'fecha_auditoria', 'empresa', 'area', 'tipo_auditoria', 'hallazgo', 'categoria_id', 'categoria', 'criterio_evaluacion', 'clasificacion', 'nivel_riesgo', 'estado', 'responsable_area', 'fecha_cierre', 'origen_dato', 'escenario_id', 'split']

Primeros registros:


,id_hallazgo,id_auditoria,fecha_auditoria,empresa,area,tipo_auditoria,hallazgo,categoria_id,categoria,criterio_evaluacion,clasificacion,nivel_riesgo,estado,responsable_area,fecha_cierre,origen_dato,escenario_id,split
0,SST-04392,AUD-0879,2024-11-04,Empresa Andina SAS,Obras,Auditoria interna SG-SST,El equipo auditor documento la siguiente situa...,CONTRATISTAS,Gestion SST de contratistas,"Verificacion previa, coordinacion, seguimiento...",No conformidad,Alto,Abierto,Obras,NaN,SINTETICO,CONTRATISTAS-05,train
1,SST-03419,AUD-0684,2024-06-09,Constructora Atlas SAS,Talento humano,Auditoria interna SG-SST,La evidencia recopilada permite establecer que...,DOCUMENTAL,Control documental SG-SST,"Versiones, aprobaciones, trazabilidad, acceso ...",Observacion,Medio,Cerrado,Talento humano,2024-10-01,SINTETICO,DOCUMENTAL-04,test
2,SST-02966,AUD-0594,2025-12-28,Constructora Atlas SAS,Servicios generales,Auditoria interna SG-SST,Al revisar las condiciones de la planta de pro...,FORMACION,Formacion y competencias SST,"Inducciones, entrenamiento y evidencia de comp...",No conformidad,Alto,Cerrado,Servicios generales,2026-02-05,SINTETICO,FORMACION-07,train
3,SST-01228,AUD-0246,2024-06-26,Constructora Atlas SAS,Obras,Auditoria interna SG-SST,"Segun la inspeccion realizada, el pasillo de c...",LOCATIVOS,Condiciones locativas,"Orden, aseo, rutas, almacenamiento, pisos y se...",No conformidad,Alto,Abierto,Obras,NaN,SINTETICO,LOCATIVOS-06,train
4,SST-02313,AUD-0463,2024-12-13,Empresa Andina SAS,Logistica,Auditoria interna SG-SST,La muestra revisada revelo que un incidente si...,INCIDENTES,Investigacion de incidentes y accidentes,"Reporte, investigacion, causalidad y seguimien...",Observacion,Medio,Cerrado,Logistica,2025-02-15,SINTETICO,INCIDENTES-00,train


In [2]:
columnas_interes = [
    "fecha_auditoria",
    "categoria",
    "hallazgo",
    "estado",
    "fecha_cierre"
]

for columna in columnas_interes:

    if columna in df.columns:

        print(f"\n--- {columna} ---")

        print(
            "Valores faltantes:",
            df[columna].isna().sum()
        )

        print(
            "Valores diferentes:",
            df[columna].nunique()
        )

        print("Ejemplos:")

        print(
            df[columna].dropna()
            .astype(str)
            .unique()[:10]
        )


--- fecha_auditoria ---
Valores faltantes: 0
Valores diferentes: 991
Ejemplos:
<StringArray>
['2024-11-04', '2024-06-09', '2025-12-28', '2024-06-26', '2024-12-13',
 '2026-02-13', '2025-08-18', '2026-03-25', '2024-07-24', '2024-09-27']
Length: 10, dtype: str

--- categoria ---
Valores faltantes: 0
Valores diferentes: 12
Ejemplos:
<StringArray>
[                       'Gestion SST de contratistas',
                          'Control documental SG-SST',
                       'Formacion y competencias SST',
                              'Condiciones locativas',
           'Investigacion de incidentes y accidentes',
                   'Elementos de proteccion personal',
              'Preparacion y respuesta a emergencias',
                       'Inspecciones y mantenimiento',
 'Identificacion de peligros y valoracion de riesgos',
                     'Gestion de sustancias quimicas']
Length: 10, dtype: str

--- hallazgo ---
Valores faltantes: 0
Valores diferentes: 3396
Ejemplos:
<String

In [3]:
# Crear copia para las consultas
auditorias = df.copy()

# Convertir fecha de auditoría
auditorias["fecha_auditoria"] = pd.to_datetime(
    auditorias["fecha_auditoria"],
    errors="coerce"
)

# Extraer año y mes
auditorias["anio"] = (
    auditorias["fecha_auditoria"].dt.year
)

auditorias["mes"] = (
    auditorias["fecha_auditoria"].dt.month
)

print("Registros por año:")

display(
    auditorias["anio"]
    .value_counts()
    .sort_index()
)

print("\nCategorías registradas:")

for categoria in sorted(
    auditorias["categoria"].dropna().unique()
):
    print("-", categoria)

print(
    "\nFechas no interpretadas:",
    auditorias["fecha_auditoria"].isna().sum()
)

Registros por año:


anio
2024    1862
2025    1781
2026    1357
Name: count, dtype: int64


Categorías registradas:
- Condiciones locativas
- Control documental SG-SST
- Elementos de proteccion personal
- Formacion y competencias SST
- Gestion SST de contratistas
- Gestion de sustancias quimicas
- Identificacion de peligros y valoracion de riesgos
- Inspecciones y mantenimiento
- Investigacion de incidentes y accidentes
- Preparacion y respuesta a emergencias
- Seguridad de maquinas y herramientas
- Trabajo en alturas

Fechas no interpretadas: 0


In [4]:
def normalizar_texto(texto):

    texto = str(texto).lower()

    texto = unicodedata.normalize(
        "NFD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

    return texto.strip()


# Equivalencias para las categorías
alias_categorias = {
    "epp": "Elementos de protección personal",
    "elementos de proteccion personal": "Elementos de protección personal",
    "proteccion personal": "Elementos de protección personal",
    "trabajo en alturas": "Trabajo en alturas",
    "trabajos en alturas": "Trabajo en alturas",
    "sustancias quimicas": "Gestión de sustancias químicas",
    "riesgos quimicos": "Gestión de sustancias químicas"
}


def interpretar_pregunta(pregunta):

    texto = normalizar_texto(pregunta)

    # Identificar intención
    if any(
        palabra in texto
        for palabra in [
            "cuantos", "cuantas",
            "cantidad", "numero", "total"
        ]
    ):
        intencion = "conteo"

    elif any(
        palabra in texto
        for palabra in [
            "mas frecuentes",
            "mas hallazgos",
            "categorias principales",
            "ranking"
        ]
    ):
        intencion = "ranking"

    else:
        intencion = "desconocida"

    # Detectar año de cuatro dígitos
    coincidencias = re.findall(
        r"\b(?:19|20)\d{2}\b",
        texto
    )

    anio = (
        int(coincidencias[0])
        if len(coincidencias) == 1
        else None
    )

    # Detectar categoría
    categoria = None

    # Añadir automáticamente los nombres del dataset
    equivalencias = {
        normalizar_texto(c): c
        for c in auditorias["categoria"].dropna().unique()
    }

    equivalencias.update(alias_categorias)

    for alias in sorted(
        equivalencias,
        key=len,
        reverse=True
    ):

        if re.search(
            rf"(?<!\w){re.escape(alias)}(?!\w)",
            texto
        ):
            categoria = equivalencias[alias]
            break

    return {
        "intencion": intencion,
        "categoria": categoria,
        "anio": anio
    }

In [5]:
preguntas = [
    "¿Cuántos hallazgos de EPP hubo en 2025?",
    "¿Cuál es el total de hallazgos en 2026?",
    "¿Cuántos hallazgos de trabajo en alturas hubo en 2024?",
    "¿Cuáles fueron las categorías con más hallazgos en 2025?"
]

for pregunta in preguntas:

    print("\nPregunta:", pregunta)

    resultado = interpretar_pregunta(pregunta)

    print("Interpretación:", resultado)


Pregunta: ¿Cuántos hallazgos de EPP hubo en 2025?
Interpretación: {'intencion': 'conteo', 'categoria': 'Elementos de protección personal', 'anio': 2025}

Pregunta: ¿Cuál es el total de hallazgos en 2026?
Interpretación: {'intencion': 'conteo', 'categoria': None, 'anio': 2026}

Pregunta: ¿Cuántos hallazgos de trabajo en alturas hubo en 2024?
Interpretación: {'intencion': 'conteo', 'categoria': 'Trabajo en alturas', 'anio': 2024}

Pregunta: ¿Cuáles fueron las categorías con más hallazgos en 2025?
Interpretación: {'intencion': 'ranking', 'categoria': None, 'anio': 2025}


In [6]:
def normalizar_texto(texto):

    texto = str(texto).lower()

    texto = unicodedata.normalize("NFD", texto)

    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )

    return texto.strip()


# Mapa de categorías reales
categorias_reales = {
    normalizar_texto(c): c
    for c in auditorias["categoria"].dropna().unique()
}

# Alias hacia el nombre normalizado
alias_categorias = {
    "epp": "elementos de proteccion personal",
    "proteccion personal": "elementos de proteccion personal",
    "trabajos en alturas": "trabajo en alturas",
    "sustancias quimicas": "gestion de sustancias quimicas",
    "riesgos quimicos": "gestion de sustancias quimicas",
    "capacitaciones": "formacion y competencias sst",
    "emergencias": "preparacion y respuesta a emergencias"
}

# Resolver los alias contra las categorías del CSV
equivalencias = dict(categorias_reales)

for alias, nombre_normalizado in alias_categorias.items():
    equivalencias[alias] = categorias_reales[nombre_normalizado]


def interpretar_pregunta(pregunta):

    texto = normalizar_texto(pregunta)

    # Reconocer primero rankings
    if any(frase in texto for frase in [
        "mas frecuentes",
        "mas hallazgos",
        "categorias principales",
        "ranking",
        "mayor cantidad"
    ]):
        intencion = "ranking"

    elif any(frase in texto for frase in [
        "cuantos",
        "cuantas",
        "cantidad",
        "numero",
        "total"
    ]):
        intencion = "conteo"

    else:
        intencion = "desconocida"

    # Identificar año
    anios = re.findall(r"\b(?:19|20)\d{2}\b", texto)

    anio = int(anios[0]) if len(anios) == 1 else None

    # Identificar categoría
    categoria = None

    for alias in sorted(equivalencias, key=len, reverse=True):

        if re.search(
            rf"(?<!\w){re.escape(alias)}(?!\w)",
            texto
        ):
            categoria = equivalencias[alias]
            break

    # Identificar estado
    estado = None

    if "en seguimiento" in texto:
        estado = "En seguimiento"
    elif "abiertos" in texto or "abiertas" in texto:
        estado = "Abierto"
    elif "cerrados" in texto or "cerradas" in texto:
        estado = "Cerrado"

    # Detectar entidad solicitada
    entidad = (
        "auditorias"
        if re.search(r"\bauditorias\b", texto)
        else "hallazgos"
    )

    return {
        "intencion": intencion,
        "categoria": categoria,
        "anio": anio,
        "estado": estado,
        "entidad": entidad
    }

In [7]:
def consultar_auditorias(pregunta):

    interpretacion = interpretar_pregunta(pregunta)

    intencion = interpretacion["intencion"]
    categoria = interpretacion["categoria"]
    anio = interpretacion["anio"]
    estado = interpretacion["estado"]
    entidad = interpretacion["entidad"]

    # No ejecutar consultas desconocidas
    if intencion == "desconocida":
        return {
            "respuesta": "No pude interpretar la consulta.",
            "interpretacion": interpretacion,
            "datos": None
        }

    # Validar que los filtros reconocidos cubran la pregunta
    texto = normalizar_texto(pregunta)

    if len(re.findall(r"\b(?:19|20)\d{2}\b", texto)) > 1:
        return {
            "respuesta": "Todavía no admito consultas de varios años.",
            "interpretacion": interpretacion,
            "datos": None
        }

    # Evitar interpretar 'pendientes' como todos los registros
    if "pendiente" in texto:
        return {
            "respuesta": (
                "La consulta sobre pendientes requiere "
                "definir los estados que incluye."
            ),
            "interpretacion": interpretacion,
            "datos": None
        }

    # Copiar información histórica
    datos = auditorias.copy()

    # Aplicar filtros
    if anio is not None:
        datos = datos[datos["anio"] == anio]

    if categoria is not None:
        datos = datos[datos["categoria"] == categoria]

    if estado is not None:
        datos = datos[datos["estado"] == estado]

    # Ejecutar operación
    if intencion == "conteo":

        if entidad == "auditorias":
            cantidad = datos["id_auditoria"].nunique()
            unidad = "auditorías diferentes"

        else:
            cantidad = len(datos)
            unidad = "hallazgos"

        respuesta = (
            f"Se encontraron {cantidad} {unidad} "
            "con los filtros indicados."
        )

        resultado = cantidad

    elif intencion == "ranking":

        if entidad != "hallazgos":
            return {
                "respuesta": (
                    "El ranking de auditorías todavía "
                    "no está implementado."
                ),
                "interpretacion": interpretacion,
                "datos": None
            }

        resultado = (
            datos.groupby("categoria")
            .size()
            .sort_values(ascending=False)
        )

        respuesta = (
            "Este es el ranking de categorías "
            "por número de hallazgos."
        )

    return {
        "respuesta": respuesta,
        "interpretacion": interpretacion,
        "datos": resultado
    }

In [8]:
preguntas_prueba = [
    "¿Cuántos hallazgos de EPP hubo en 2025?",
    "¿Cuál es el total de hallazgos en 2026?",
    "¿Cuántos hallazgos de trabajo en alturas hubo en 2024?",
    "¿Cuáles fueron las categorías con más hallazgos en 2025?",
    "¿Cuántos hallazgos abiertos hubo en 2025?",
    "¿Cuántas auditorías hubo en 2024?"
]

for pregunta in preguntas_prueba:

    resultado = consultar_auditorias(pregunta)

    print("=" * 65)
    print("PREGUNTA:", pregunta)
    print("INTERPRETACIÓN:", resultado["interpretacion"])
    print("RESPUESTA:", resultado["respuesta"])

    if isinstance(resultado["datos"], pd.Series):
        display(resultado["datos"].head(5))

PREGUNTA: ¿Cuántos hallazgos de EPP hubo en 2025?
INTERPRETACIÓN: {'intencion': 'conteo', 'categoria': 'Elementos de proteccion personal', 'anio': 2025, 'estado': None, 'entidad': 'hallazgos'}
RESPUESTA: Se encontraron 143 hallazgos con los filtros indicados.
PREGUNTA: ¿Cuál es el total de hallazgos en 2026?
INTERPRETACIÓN: {'intencion': 'conteo', 'categoria': None, 'anio': 2026, 'estado': None, 'entidad': 'hallazgos'}
RESPUESTA: Se encontraron 1357 hallazgos con los filtros indicados.
PREGUNTA: ¿Cuántos hallazgos de trabajo en alturas hubo en 2024?
INTERPRETACIÓN: {'intencion': 'conteo', 'categoria': 'Trabajo en alturas', 'anio': 2024, 'estado': None, 'entidad': 'hallazgos'}
RESPUESTA: Se encontraron 144 hallazgos con los filtros indicados.
PREGUNTA: ¿Cuáles fueron las categorías con más hallazgos en 2025?
INTERPRETACIÓN: {'intencion': 'ranking', 'categoria': None, 'anio': 2025, 'estado': None, 'entidad': 'hallazgos'}
RESPUESTA: Este es el ranking de categorías por número de hallazgos

categoria
Formacion y competencias SST                          159
Trabajo en alturas                                    156
Preparacion y respuesta a emergencias                 156
Inspecciones y mantenimiento                          155
Identificacion de peligros y valoracion de riesgos    153
dtype: int64

PREGUNTA: ¿Cuántos hallazgos abiertos hubo en 2025?
INTERPRETACIÓN: {'intencion': 'conteo', 'categoria': None, 'anio': 2025, 'estado': 'Abierto', 'entidad': 'hallazgos'}
RESPUESTA: Se encontraron 395 hallazgos con los filtros indicados.
PREGUNTA: ¿Cuántas auditorías hubo en 2024?
INTERPRETACIÓN: {'intencion': 'conteo', 'categoria': None, 'anio': 2024, 'estado': None, 'entidad': 'auditorias'}
RESPUESTA: Se encontraron 901 auditorías diferentes con los filtros indicados.


In [9]:
# Casos con resultados que podemos verificar directamente

casos = [
    (
        "¿Cuál es el total de hallazgos en 2026?",
        len(auditorias[auditorias["anio"] == 2026])
    ),
    (
        "¿Cuántos hallazgos de EPP hubo en 2025?",
        len(
            auditorias[
                (auditorias["anio"] == 2025) &
                (
                    auditorias["categoria"] ==
                    categorias_reales[
                        "elementos de proteccion personal"
                    ]
                )
            ]
        )
    ),
    (
        "¿Cuántas auditorías hubo en 2024?",
        auditorias.loc[
            auditorias["anio"] == 2024,
            "id_auditoria"
        ].nunique()
    )
]

for pregunta, esperado in casos:

    obtenido = consultar_auditorias(pregunta)["datos"]

    assert obtenido == esperado, (
        f"Error en: {pregunta}. "
        f"Esperado: {esperado}, obtenido: {obtenido}"
    )

    print(f"OK | {pregunta} | Resultado: {obtenido}")

print("\nTodas las pruebas pasaron correctamente.")

OK | ¿Cuál es el total de hallazgos en 2026? | Resultado: 1357
OK | ¿Cuántos hallazgos de EPP hubo en 2025? | Resultado: 143
OK | ¿Cuántas auditorías hubo en 2024? | Resultado: 901

Todas las pruebas pasaron correctamente.


In [10]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

modelo_nlp = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)

print("Modelo semántico cargado correctamente")
print("Dispositivo:", device)
print("Dimensiones:", modelo_nlp.get_sentence_embedding_dimension())

d:\VIVIAN\Uis\Quinto semestre\Deep Learning\Proyecto Mini-Transforms\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1792.79it/s]


Modelo semántico cargado correctamente
Dispositivo: cuda
Dimensiones: 384


C:\Users\MSI\AppData\Local\Temp\ipykernel_3356\3540638340.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensiones:", modelo_nlp.get_sentence_embedding_dimension())


In [11]:
intenciones = {
    "conteo": [
        "¿Cuántos hallazgos se registraron?",
        "Necesito conocer la cantidad de incumplimientos.",
        "Dime el número total de registros.",
        "¿Qué cantidad de observaciones se encontraron?",
        "Quiero saber cuántos casos existen."
    ],

    "ranking": [
        "¿Cuáles son las categorías con más hallazgos?",
        "¿Qué tipos de incumplimientos son los más frecuentes?",
        "Ordena las categorías según la cantidad de casos.",
        "¿Dónde se concentran más problemas?",
        "Muéstrame los principales tipos de hallazgos."
    ],

    "comparacion_anual": [
        "¿En qué año hubo más hallazgos?",
        "Compara los incumplimientos entre diferentes años.",
        "¿Cómo cambió la cantidad de casos de un año a otro?",
        "Quiero comparar los registros históricos por año.",
        "¿Cuál fue el año con mayor número de observaciones?"
    ],

    "listado": [
        "Muéstrame los hallazgos registrados.",
        "Necesito ver los registros de auditoría.",
        "Dame una lista de las observaciones encontradas.",
        "¿Qué hallazgos se documentaron?",
        "Quiero consultar los detalles de los casos."
    ]
}

# Organizar los ejemplos
textos_referencia = []
etiquetas_referencia = []

for intencion, ejemplos in intenciones.items():
    for ejemplo in ejemplos:
        textos_referencia.append(ejemplo)
        etiquetas_referencia.append(intencion)

# Generar embeddings con MiniLM
embeddings_referencia = modelo_nlp.encode(
    textos_referencia,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Intenciones disponibles:", len(intenciones))
print("Ejemplos de referencia:", len(textos_referencia))
print("Forma de embeddings:", embeddings_referencia.shape)

Intenciones disponibles: 4
Ejemplos de referencia: 20
Forma de embeddings: torch.Size([20, 384])


In [12]:
from sentence_transformers import util
import pandas as pd

def interpretar_con_ia(pregunta):

    # Convertir la pregunta a embedding
    embedding_pregunta = modelo_nlp.encode(
        pregunta,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    # Calcular similitud semántica
    similitudes = util.cos_sim(
        embedding_pregunta,
        embeddings_referencia
    )[0]

    # Guardar resultados por intención
    resultados = {}

    for intencion in intenciones:

        indices = [
            i for i, etiqueta in enumerate(etiquetas_referencia)
            if etiqueta == intencion
        ]

        puntuaciones = similitudes[indices]

        resultados[intencion] = float(
            puntuaciones.max().item()
        )

    # Seleccionar la intención más similar
    ranking = sorted(
        resultados.items(),
        key=lambda elemento: elemento[1],
        reverse=True
    )

    mejor_intencion = ranking[0][0]
    mejor_similitud = ranking[0][1]

    return {
        "pregunta": pregunta,
        "intencion_predicha": mejor_intencion,
        "similitud": round(mejor_similitud, 4),
        "ranking": ranking
    }

In [13]:
preguntas_nuevas = [
    "¿Qué tanto se reportó durante 2025?",
    "¿Dónde encontramos la mayor concentración de problemas?",
    "¿Cuál fue el periodo anual con más incumplimientos?",
    "Enséñame los casos que quedaron documentados.",
    "¿En qué categorías se repitieron más las fallas?",
    "Quisiera consultar todas las observaciones realizadas.",
    "¿Se presentaron más problemas en 2024 o en 2025?",
    "¿Qué cantidad de situaciones fueron reportadas?"
]

resultados_ia = []

for pregunta in preguntas_nuevas:

    resultado = interpretar_con_ia(pregunta)

    resultados_ia.append({
        "pregunta": pregunta,
        "intencion": resultado["intencion_predicha"],
        "similitud": resultado["similitud"]
    })

tabla_resultados = pd.DataFrame(resultados_ia)

display(tabla_resultados)

,pregunta,intencion,similitud
0,¿Qué tanto se reportó durante 2025?,comparacion_anual,0.5675
1,¿Dónde encontramos la mayor concentración de p...,ranking,0.8456
2,¿Cuál fue el periodo anual con más incumplimie...,comparacion_anual,0.8127
3,Enséñame los casos que quedaron documentados.,listado,0.7290
4,¿En qué categorías se repitieron más las fallas?,ranking,0.6628
5,Quisiera consultar todas las observaciones rea...,listado,0.7176
6,¿Se presentaron más problemas en 2024 o en 2025?,ranking,0.5549
7,¿Qué cantidad de situaciones fueron reportadas?,conteo,0.7706


In [14]:
pregunta = (
    "¿En qué año encontramos más problemas "
    "relacionados con la seguridad?"
)

resultado = interpretar_con_ia(pregunta)

print("Pregunta:", resultado["pregunta"])
print("Intención predicha:", resultado["intencion_predicha"])

print("\nPuntuaciones por intención:")

for intencion, similitud in resultado["ranking"]:

    print(
        f"{intencion:20s} | "
        f"{similitud:.4f}"
    )

Pregunta: ¿En qué año encontramos más problemas relacionados con la seguridad?
Intención predicha: comparacion_anual

Puntuaciones por intención:
comparacion_anual    | 0.5917
ranking              | 0.5213
conteo               | 0.4284
listado              | 0.3595


In [15]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Preguntas nuevas que no aparecen
# en nuestros ejemplos de referencia

evaluacion = [
    ("¿Cuántas observaciones existen?", "conteo"),
    ("Necesito conocer el volumen de hallazgos.", "conteo"),
    ("¿Qué número de casos se documentó?", "conteo"),
    ("¿Qué tanto se reportó durante 2025?", "conteo"),

    ("¿Qué categorías acumulan más registros?", "ranking"),
    ("Clasifica las causas según su frecuencia.", "ranking"),
    ("¿Cuáles son los problemas más recurrentes?", "ranking"),
    ("¿Qué tipos de hallazgos predominan?", "ranking"),

    ("¿Se presentaron más problemas en 2024 o en 2025?", "comparacion_anual"),
    ("¿Cuál de los años tuvo más casos?", "comparacion_anual"),
    ("¿Aumentaron los hallazgos entre 2024 y 2025?", "comparacion_anual"),
    ("Muéstrame la evolución anual de los registros.", "comparacion_anual"),

    ("Necesito revisar los casos registrados.", "listado"),
    ("Enséñame las observaciones documentadas.", "listado"),
    ("Quiero visualizar los detalles de los hallazgos.", "listado"),
    ("¿Qué situaciones quedaron registradas?", "listado")
]

resultados_evaluacion = []

for pregunta, etiqueta_real in evaluacion:

    resultado = interpretar_con_ia(pregunta)

    resultados_evaluacion.append({
        "pregunta": pregunta,
        "real": etiqueta_real,
        "predicha": resultado["intencion_predicha"],
        "similitud": resultado["similitud"],
        "correcto": (
            etiqueta_real == resultado["intencion_predicha"]
        )
    })

df_evaluacion = pd.DataFrame(resultados_evaluacion)

display(df_evaluacion)

accuracy_ia = accuracy_score(
    df_evaluacion["real"],
    df_evaluacion["predicha"]
)

print(f"\nAccuracy: {accuracy_ia:.2%}")

print("\nREPORTE DE CLASIFICACIÓN")

print(classification_report(
    df_evaluacion["real"],
    df_evaluacion["predicha"],
    zero_division=0
))

,pregunta,real,predicha,similitud,correcto
0,¿Cuántas observaciones existen?,conteo,conteo,0.8877,True
1,Necesito conocer el volumen de hallazgos.,conteo,conteo,0.8382,True
2,¿Qué número de casos se documentó?,conteo,conteo,0.8548,True
3,¿Qué tanto se reportó durante 2025?,conteo,comparacion_anual,0.5675,False
4,¿Qué categorías acumulan más registros?,ranking,ranking,0.7069,True
5,Clasifica las causas según su frecuencia.,ranking,ranking,0.5685,True
6,¿Cuáles son los problemas más recurrentes?,ranking,ranking,0.7260,True
7,¿Qué tipos de hallazgos predominan?,ranking,ranking,0.8274,True
8,¿Se presentaron más problemas en 2024 o en 2025?,comparacion_anual,ranking,0.5549,False
9,¿Cuál de los años tuvo más casos?,comparacion_anual,comparacion_anual,0.7572,True



Accuracy: 81.25%

REPORTE DE CLASIFICACIÓN
                   precision    recall  f1-score   support

comparacion_anual       0.75      0.75      0.75         4
           conteo       1.00      0.75      0.86         4
          listado       1.00      0.75      0.86         4
          ranking       0.67      1.00      0.80         4

         accuracy                           0.81        16
        macro avg       0.85      0.81      0.82        16
     weighted avg       0.85      0.81      0.82        16



In [16]:
analisis_confianza = []

for pregunta, etiqueta_real in evaluacion:

    resultado = interpretar_con_ia(pregunta)

    ranking = resultado["ranking"]

    mejor = ranking[0][1]
    segundo = ranking[1][1]

    analisis_confianza.append({
        "pregunta": pregunta,
        "real": etiqueta_real,
        "predicha": ranking[0][0],
        "similitud": mejor,
        "margen": mejor - segundo,
        "correcto": ranking[0][0] == etiqueta_real
    })

df_confianza = pd.DataFrame(analisis_confianza)

print("Preguntas con mayor incertidumbre relativa:")

display(
    df_confianza.sort_values(
        "margen",
        ascending=True
    )
)

print("\nMargen promedio en aciertos:")
print(
    df_confianza.loc[
        df_confianza["correcto"], "margen"
    ].mean()
)

print("\nMargen promedio en errores:")
print(
    df_confianza.loc[
        ~df_confianza["correcto"], "margen"
    ].mean()
)

Preguntas con mayor incertidumbre relativa:


,pregunta,real,predicha,similitud,margen,correcto
15,¿Qué situaciones quedaron registradas?,listado,listado,0.631477,0.010439,True
7,¿Qué tipos de hallazgos predominan?,ranking,ranking,0.827397,0.011589,True
8,¿Se presentaron más problemas en 2024 o en 2025?,comparacion_anual,ranking,0.554876,0.012501,False
1,Necesito conocer el volumen de hallazgos.,conteo,conteo,0.838215,0.026004,True
14,Quiero visualizar los detalles de los hallazgos.,listado,ranking,0.805365,0.027707,False
3,¿Qué tanto se reportó durante 2025?,conteo,comparacion_anual,0.567532,0.087283,False
5,Clasifica las causas según su frecuencia.,ranking,ranking,0.568477,0.087345,True
9,¿Cuál de los años tuvo más casos?,comparacion_anual,comparacion_anual,0.757189,0.103502,True
10,¿Aumentaron los hallazgos entre 2024 y 2025?,comparacion_anual,comparacion_anual,0.593145,0.111893,True
13,Enséñame las observaciones documentadas.,listado,listado,0.857125,0.116293,True



Margen promedio en aciertos:
0.11574116807717544

Margen promedio en errores:
0.04249707857767741


In [17]:
# Descripciones del dominio SG-SST
descripciones = {
    "Elementos de proteccion personal":
        "Uso, entrega, mantenimiento y disponibilidad de cascos, "
        "guantes, gafas, respiradores y otros equipos de protección.",

    "Trabajo en alturas":
        "Prevención de caídas, arneses, líneas de vida, "
        "andamios y trabajos realizados en altura.",

    "Condiciones locativas":
        "Estado de pisos, pasillos, escaleras, señalización, "
        "orden, aseo y condiciones de las instalaciones.",

    "Formacion y competencias SST":
        "Capacitaciones, inducciones, reinducciones, "
        "entrenamientos y competencias de los trabajadores.",

    "Gestion de sustancias quimicas":
        "Manejo, almacenamiento, etiquetado, exposición "
        "y fichas de datos de seguridad de productos químicos.",

    "Seguridad de maquinas y herramientas":
        "Operación segura de maquinaria, herramientas eléctricas, "
        "guardas de seguridad y dispositivos de protección.",

    "Inspecciones y mantenimiento":
        "Inspecciones programadas, mantenimiento preventivo, "
        "revisiones técnicas y seguimiento de equipos.",

    "Preparacion y respuesta a emergencias":
        "Planes de emergencia, evacuación, simulacros, "
        "brigadas, extintores y atención de emergencias.",

    "Investigacion de incidentes y accidentes":
        "Reporte e investigación de accidentes laborales, "
        "incidentes, causas y acciones correctivas.",

    "Identificacion de peligros y valoracion de riesgos":
        "Identificación de peligros laborales, evaluación "
        "de riesgos y definición de medidas de control.",

    "Control documental SG-SST":
        "Control de documentos, procedimientos, registros, "
        "versiones, aprobaciones y evidencias del SG-SST.",

    "Gestion SST de contratistas":
        "Verificación de requisitos, documentación, "
        "seguimiento y coordinación de seguridad de contratistas."
}

# Utilizar los nombres exactos presentes en el dataset
categorias_dataset = sorted(
    auditorias["categoria"].unique()
)

assert set(categorias_dataset) == set(descripciones), (
    "Las descripciones no coinciden con las categorías del CSV."
)

textos_categorias = [
    f"{categoria}. {descripciones[categoria]}"
    for categoria in categorias_dataset
]

embeddings_categorias = modelo_nlp.encode(
    textos_categorias,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print("Categorías representadas:", len(categorias_dataset))
print("Dimensiones:", embeddings_categorias.shape)

Categorías representadas: 12
Dimensiones: torch.Size([12, 384])


In [18]:
from sentence_transformers import util

def reconocer_categoria_ia(texto, top_k=3):

    embedding = modelo_nlp.encode(
        texto,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    similitudes = util.cos_sim(
        embedding,
        embeddings_categorias
    )[0]

    valores, indices = torch.topk(
        similitudes,
        k=min(top_k, len(categorias_dataset))
    )

    resultados = []

    for valor, indice in zip(valores, indices):

        resultados.append({
            "categoria": categorias_dataset[indice.item()],
            "similitud": round(valor.item(), 4)
        })

    return resultados

In [19]:
consultas = [
    "Los trabajadores no estaban utilizando sus gafas de seguridad",
    "Problemas con arneses y líneas de vida",
    "No se realizaron entrenamientos al personal nuevo",
    "Productos peligrosos almacenados sin identificación",
    "Documentos del sistema de gestión sin actualizar",
    "¿Cuántos hallazgos de EPP hubo en 2025?",
    "¿Cuántos hallazgos hubo en 2025?"
]

for consulta in consultas:

    print("\nCONSULTA:", consulta)

    resultados = reconocer_categoria_ia(consulta)

    for resultado in resultados:
        print(
            resultado["categoria"],
            "| Similitud:",
            resultado["similitud"]
        )


CONSULTA: Los trabajadores no estaban utilizando sus gafas de seguridad
Seguridad de maquinas y herramientas | Similitud: 0.2901
Elementos de proteccion personal | Similitud: 0.2391
Inspecciones y mantenimiento | Similitud: 0.2335

CONSULTA: Problemas con arneses y líneas de vida
Condiciones locativas | Similitud: 0.3341
Trabajo en alturas | Similitud: 0.324
Preparacion y respuesta a emergencias | Similitud: 0.2512

CONSULTA: No se realizaron entrenamientos al personal nuevo
Formacion y competencias SST | Similitud: 0.4177
Inspecciones y mantenimiento | Similitud: 0.1575
Gestion SST de contratistas | Similitud: 0.1426

CONSULTA: Productos peligrosos almacenados sin identificación
Gestion de sustancias quimicas | Similitud: 0.6481
Seguridad de maquinas y herramientas | Similitud: 0.4914
Elementos de proteccion personal | Similitud: 0.4312

CONSULTA: Documentos del sistema de gestión sin actualizar
Control documental SG-SST | Similitud: 0.407
Gestion SST de contratistas | Similitud: 0.3